In [3]:
# ── Imports & paths ────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from linearmodels.panel import PanelOLS
import statsmodels.api as sm

# Table layout constants
BG     = 'white'
FG     = 'black'
ACCENT = '#444444'
RULE   = '#aaaaaa'
COLS   = [0.04, 0.50, 0.65, 0.80, 0.92]
ALIGNS = ['left', 'right', 'right', 'right', 'right']
Y0     = 0.82
ROW_H  = 0.092


# ── Helper functions ───────────────────────────────────────────────────────

def stars(p):
    """Return significance stars under the *** p<0.01, ** p<0.05, * p<0.1 convention."""
    if p < 0.01: return '***'
    if p < 0.05: return '**'
    if p < 0.10: return '*'
    return ''


def table_cell(ax, y, vals, bold=False, color=FG, small=False):
    """Render a single row of monospaced text across COLS positions."""
    fs = 9.5 if not small else 8.5
    fw = 'bold' if bold else 'normal'
    for x, txt, ha in zip(COLS, vals, ALIGNS):
        ax.text(x, y, txt, transform=ax.transAxes,
                color=color, fontsize=fs, fontweight=fw,
                va='top', ha=ha, fontfamily='monospace')


def hline(ax, y, lw=0.8, color=RULE):
    """Draw a horizontal rule spanning the table width."""
    ax.plot([0.03, 0.97], [y, y], color=color, linewidth=lw,
            transform=ax.transAxes, clip_on=False)


def make_regression_table(res, out_path):
    """Render PanelOLS results as a formatted PNG table."""
    params = res.params
    ses    = res.std_errors
    pvals  = res.pvalues
    n      = int(res.nobs)
    r2w    = res.rsquared
    fp     = res.f_statistic.pval
    fstat  = res.f_statistic.stat

    fig_t, ax_t = plt.subplots(figsize=(10, 5.8))
    fig_t.patch.set_facecolor(BG)
    ax_t.set_facecolor(BG)
    ax_t.set_axis_off()
    ax_t.set_xlim(0, 1)
    ax_t.set_ylim(0, 1)

    ax_t.text(0.5, 0.97,
              'Aid Spending and Net Displacement Flow\n(Province FE, Robust SE)',
              transform=ax_t.transAxes, color=FG, fontsize=11.5,
              fontweight='bold', ha='center', va='top')

    y = Y0
    hline(ax_t, y, lw=1.2, color='black')
    y -= 0.01

    table_cell(ax_t, y, ['', 'Coef.', 'p-value', 'Std. Err.', 't-stat'], bold=True, color=ACCENT)
    y -= ROW_H
    hline(ax_t, y, lw=0.6)
    y -= 0.01

    coef_rows = [
        ('Intercept',              'const'),
        ('Log new project spend',  'log_new_project_spend'),
        ('Log total active spend', 'log_total_active_spend'),
    ]
    for label, key in coef_rows:
        coef = params[key]
        se   = ses[key]
        t    = coef / se
        p    = pvals[key]
        table_cell(ax_t, y, [
            label,
            f'{coef:,.0f}{stars(p)}',
            f'{p:.3f}',
            f'({se:,.0f})',
            f'{t:.3f}',
        ])
        y -= ROW_H

    hline(ax_t, y, lw=0.6)
    y -= 0.01

    table_cell(ax_t, y, ['N',            str(n),         '', '', ''])
    y -= ROW_H * 0.82
    table_cell(ax_t, y, ['R² (within)', f'{r2w:.3f}',    '', '', ''])
    y -= ROW_H * 0.82
    table_cell(ax_t, y, ['F-statistic',  f'{fstat:.3f}', f'p = {fp:.3f}', '', ''])
    y -= ROW_H * 0.82
    table_cell(ax_t, y, ['Entity FE',   'Yes',           '', '', ''])
    y -= ROW_H * 0.82

    hline(ax_t, y, lw=1.2, color='black')
    y -= 0.01

    ax_t.text(0.04, y,
              'Robust standard errors in parentheses.  *** p<0.01  ** p<0.05  * p<0.1\n'
              'Dependent variable: net monthly displacement flow (displaced − returnees).\n'
              'Sample: Nord-Kivu, Sud-Kivu, Ituri provinces.',
              transform=ax_t.transAxes, color=ACCENT, fontsize=8.0,
              va='top', fontfamily='monospace')

    fig_t.savefig(out_path, dpi=150, bbox_inches='tight', facecolor=BG)
    plt.close(fig_t)
    print(f'Saved → {out_path}')


In [ ]:
# ── Load merged panel ──────────────────────────────────────────────────────
DATA_DIR = '/Users/jackzipper/QSS20/final_project/final_project_data/'
OUT_DIR  = '/Users/jackzipper/QSS20/final_project/output/'

df = pd.read_csv(DATA_DIR + 'aid_displacement_merged.csv')
df['snapshot_month'] = pd.to_datetime(df['snapshot_month'])
print(f'Loaded {len(df):,} rows from aid_displacement_merged.csv')

# ── Fit PanelOLS: entity FE, robust SE ────────────────────────────────────
df_panel = df.set_index(['admin1_label', 'snapshot_month'])

mod = PanelOLS(
    dependent      = df_panel['net_monthly_flow'],
    exog           = sm.add_constant(df_panel[['log_new_project_spend', 'log_total_active_spend']]),
    entity_effects = True,
    time_effects   = False,
)
res = mod.fit(cov_type='robust')
print(res.summary)


In [ ]:
# ── Export regression table PNG ────────────────────────────────────────────
make_regression_table(res, OUT_DIR + 'aid_displacement_regression.png')
